# 03 · Filter & Rank — run the shared multi-layer filter (`design_type="antibody"`)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 17** you map your VHH campaign onto `fp.Design` objects, run the pipeline with the
**antibody** cutoffs (scRMSD ≤ 3.0, pLDDT ≥ 70, pae_interaction ≤ 12), and report survival (D3 pt 1).

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline

This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"antibody"` cutoffs.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)
print("\nUsing design_type='antibody':", fp.DEFAULT_CUTOFFS["antibody"])

## Build `fp.Design` objects from the campaign

The filter operates on `fp.Design` records. Map each VHH's AF2-Multimer-ab metrics onto the matching
fields. For an antibody complex the key fields are `plddt`, `pae_interaction`, and `scrmsd`; we stash
the developability heuristics + CDR3 length in `extra` so they ride along into the ranked CSV.

In [ ]:
camp = pd.read_csv("results/campaign.csv")

designs = []
for _, r in camp.iterrows():
    designs.append(fp.Design(
        design_id=str(r["design_id"]),
        sequence="",                      # full VHH not needed for the confidence layers
        design_type="antibody",
        plddt=r.get("plddt"),
        pae_interaction=r.get("pae_interaction"),
        scrmsd=r.get("scrmsd"),
        # solubility maps to the CamSol-like heuristic so Layer 3 (physics) has something to act on:
        solubility=r.get("camsol_like"),
        extra={"tap_score": r.get("tap_score"), "humanness": r.get("humanness"),
               "cdr_geom": r.get("cdr_geom"), "cdr3_len": r.get("cdr3_len"),
               "synthetic": bool(r.get("synthetic", False))},
    ))
print(len(designs), "fp.Design objects built (design_type='antibody')")

## Run the pipeline + report

`run_pipeline(..., design_type="antibody")` applies the layers in order with the antibody cutoffs and
returns a ranked DataFrame; `report()` prints the **survival-at-each-layer** accounting and saves the
ranked CSV + figure. We run Layers 1 + 3 here (self-consistency + physics/solubility); Layer 2
(orthogonal predictor agreement) needs a second predictor — wire IgFold/ESMFold scRMSD in for the real
run. **On mock data the survivors are SYNTHETIC** — the point is the plumbing and the honest accounting.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="antibody", use_layers=(1, 3))
top = fp.report(df_ranked, top_n=15, save_prefix="results/proj17")
print("\nranked CSV -> results/proj17_ranked.csv ; survival figure -> results/proj17_survival.png")
top

## Hit-rate accounting (report the rate, not the cherry)

Survival = how many of the generated pool pass each layer. For de novo nanobodies this is **low by
design** — that is the honest, expected result, and it is exactly why survivors go to a display screen
rather than straight to characterization.

In [ ]:
import pandas as pd
n_total = df_ranked.attrs.get("n_total", len(df_ranked))
survival = df_ranked.attrs.get("survival", {})
print(f"Generated: {n_total}")
for layer, n in survival.items():
    print(f"  {layer}: {n} survivors ({100*n/max(n_total,1):.1f}%)")
print("\nlayers_passed distribution:")
if "layers_passed" in df_ranked:
    print(df_ranked["layers_passed"].value_counts().sort_index())
print("\nReminder: mock survivors are SYNTHETIC. On a real campaign, expect a LOW pass rate — "
      "frame survivors as display-screen inputs, not finished binders.")

## D3 (part 1) checklist
- [ ] `results/proj17_ranked.csv` produced by the **shared** module with `design_type="antibody"`.
- [ ] Survival-at-each-layer reported (the survival figure saved).
- [ ] Mapping assumptions written down (which metric → which `fp.Design` field).
- [ ] Honest hit-rate accounting; survivors framed as screening inputs.

**Next:** `04_validate.ipynb` — epitope-choice + receptor-family specificity + developability figures.